<div class="row">
  <div class="column">
    <img src="./img/logo-onera.png" width="200">
  </div>
  <div class="column">
    <img src="./img/logo-ISAE_SUPAERO.png" width="200">
  </div>
</div>

# Mission with HE power train

The following Notebook shows the user an example of how to use the newly developed power train builder along with the mission from FAST-OAD_CS23.

In [ ]:
import os.path as pth
from shutil import rmtree
import logging

import pytest

import openmdao.api as om
import fastoad.api as oad
import fastga.utils.postprocessing.analysis_and_plots as api_plots

from fastga.utils.postprocessing.analysis_and_plots import (
    mass_breakdown_bar_plot,
    aircraft_geometry_plot,
)

from utils.filter_residuals import filter_residuals

from fastga_he.gui.power_train_network_viewer import power_train_network_viewer
from fastga_he.gui.power_train_weight_breakdown import power_train_mass_breakdown

#new
import pandas as pd
from bs4 import BeautifulSoup
import xml.etree.cElementTree as et




#Ta com erro no calculo de inverter mass e harness. Falta calcular as massas de propeller_1 e propeller_elec e dos fuel system

#Densidades de bateria:

densities = [350.0] # [300.0, 350.0, 400.0, 500.0]

#Dados de range de estudo

distances = [250.0] #[100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0, 500.0]

#Baselines para comaparação:

base_150 = ["-","-","-" ,255.05,3.07,972.80,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]

base_250 = ["-","-","-" ,349.60,2.53,1333.40,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]

base_500 = ["-","-","-" ,590.13,2.13,2250.80,0,2378.74,1140.00,2270.00,"-","-","-","-","-","-","-","-","-"]
 
baseline = {250: base_250} #na real que é 150, 250 e 500 baseline = {100: base_100,150: base_150, 200: base_200}
#{100.0: base_150, 150: base_150, 200: base_150, 250: base_250, 300: base_250, 350: base_250, 400: base_500, 450: base_500, 500: base_500}

#Taxa de hibridização

n = 2 #numero de motores eletricos. MODIFICAR COM O NUMERO DE MOTORES

hibrid = [0.10] #para 4 motores deu pau com 0.25. Testar com 0.2 pra ver até onde ele vai e dps retestar com 0.25 pra tentar descobrir o erro. [0.0, 0.05, 0.1, 0.15, 0.20, 0.25]

#Rodou tudo com He = 0.2. Vamo testar para 2 motores

#Para 2 motores, deu pau com He = 0.15.

#Rodou com He = 0.1

#Criação do DataFrame para os dados

df = pd.DataFrame()

DATA_FOLDER_PATH =  "data"
RESULTS_FOLDER_PATH = "results"

for rho in densities:
    for dist in distances:

        datum = {"rho_bat": [baseline[dist][0]],"range_nmi": ["Base" + " " + str(dist)], "he": [baseline[dist][2]] , "fuel_mass": [baseline[dist][3]], "emission_factor": [baseline[dist][4]], "total_emissions": [baseline[dist][5]] ,"red_emission": [baseline[dist][6]], "OWE": [baseline[dist][7]], "payload": [baseline[dist][8]], "empty_mass": [baseline[dist][9]], "he_mass": [baseline[dist][10]], "dc_dc_mass": [baseline[dist][11]], "dc_bus_mass": [baseline[dist][12]], "dc_cable_harn": [baseline[dist][13]], "motor_mass": [baseline[dist][14]], "battery_mass": [baseline[dist][15]], "fuel_flowed": [baseline[dist][16]], "inverter_mass": [baseline[dist][17]], "turboshaft_mass": [baseline[dist][18]]}

        for he in hibrid:
            
            #A posição disso talvez tenha que mudar...
            #Modificação do Input, foi o que a gente fez antes...
            """06/12/2025"""
            #Modificar Input_CaravanDEP ao inves de oad_process_inputs_CaravanDEP_retrofit → isso sera feito automatico com a mudança de Input_CaravanDEP.   
            with open(r".\data\Input_CaravanDEP_2_036.xml", "r", encoding="utf-8") as f: #MODIFICAR COM O NUMERO DE MOTORES
                soup = BeautifulSoup(f, "xml")  # use "xml" parser for XML mode

            range_tag = soup.find("TLAR").find("range")
            print("Old value:", range_tag.text, range_tag.get("units"))

            # Modify the value and the units
            range_tag.string = str(dist)
            range_tag["units"] = "nmi"

            # Modification for thrust vector

            thrust_tag = soup.find('thrust_distribution')

            thrust_part = float(he/n/(1.0 - he))

            thrust_tag.string = str([1.0] + [thrust_part]*n) #Ta automatico agora. Nao precisa mudar pra cada caso
            
            #str([1.0, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part]) #MODIFICAR COM O NUMERO DE MOTORES N thrust_parts

            # 2 motores str([1.0, thrust_part, thrust_part])
            # 4 motores str([1.0, thrust_part, thrust_part, thrust_part, thrust_part])
            # 6 motores str([1.0, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part, thrust_part])

            print("Pwr density: ", rho)
            print("Dist value:", float(range_tag.text), range_tag.get("units"))
            print("HE level: ", he)

            #Save do input modificado
            #Mudar o nome aqui
            with open("./data/Input_CaravanDEP_2_036.xml", "w", encoding="utf-8") as f:
                f.write(str(soup))
                
            def test_retrofit_DEP():
                logging.basicConfig(level=logging.WARNING)
                logging.getLogger("fastoad.module_management._bundle_loader").disabled = True
                logging.getLogger("fastoad.openmdao.variables.variable").disabled = True

                # Define used files depending on options
                #MODIFICAR COM O NUMERO DE MOTORES
                xml_file_name = "input_CaravanDEP_2_036.xml"
                process_file_name = "CaravanDEP_retrofit.yml" #nao muda

                configurator = oad.FASTOADProblemConfigurator(pth.join(DATA_FOLDER_PATH, process_file_name))
                problem = configurator.get_problem()

                # Create inputs
                ref_inputs = pth.join(DATA_FOLDER_PATH, xml_file_name)

                problem.model_options["*propeller_1*"] = {"mass_as_input": True}
                problem.model_options["*"] = {"cell_weight_ref": 50.0e-3 * 261.0 / rho}

                problem.write_needed_inputs(ref_inputs)
                problem.read_inputs()
                problem.setup()
                
                # om.n2(problem)

                problem.run_model()

                _, _, residuals = problem.model.get_nonlinear_vectors()
                residuals = filter_residuals(residuals)

                problem.write_outputs()

                # deltaT = problem.get_val("performances.solve_equilibrium.compute_dep_equilibrium.compute_energy_consumed.power_train_performances.harness_1.temperature_profile.temperature.temperature_increase.cable_temperature_increase", units="K")
                # print(deltaT)

            def test_CaravanDEP_mass_breakdown():
                #MODIFICAR COM O NUMERO DE MOTORES
                path_to_result_file = pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit_2_036.xml") 
                path_to_pt_file = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit_2.yml")

                fig = power_train_mass_breakdown(path_to_result_file, path_to_pt_file)
                fig.update_layout(uniformtext=dict(minsize=28))
                fig.update_traces(textfont=dict(family=["Arial Black", "Arial"], size=[30]))
                fig.show()
                fig = api_plots.mass_breakdown_bar_plot(path_to_result_file)
                fig.show()


            test_retrofit_DEP()
            test_CaravanDEP_mass_breakdown()

In [ ]:

# Diagnostic: Check where the module is being imported from
import fastga_he.models.propulsion.components.loads.pmsm.components.sizing_diameter_scaling as sds_module
print("Module location:", sds_module.__file__)

# Check the source code
import inspect
print("\nCurrent compute method source:")
print(inspect.getsource(sds_module.SizingMotorDiameterScaling.compute))


In [ ]:
import os.path as pth
from shutil import rmtree
import logging

import pytest

import openmdao.api as om
import fastoad.api as oad
import fastga.utils.postprocessing.analysis_and_plots as api_plots

from fastga.utils.postprocessing.analysis_and_plots import (
    mass_breakdown_bar_plot,
    aircraft_geometry_plot,
)

fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_CaravanDEP_retrofit.xml"), name="")
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec_CaravanDEP_retrofit.xml"), name="", fig = fig)
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec2_CaravanDEP_retrofit.xml"), name="", fig=fig)
fig = api_plots.aircraft_geometry_plot(pth.join(RESULTS_FOLDER_PATH, "oad_process_outputs_elec3_CaravanDEP_retrofit.xml"), name="", fig=fig)
fig.show()

In [ ]:
import os.path as pth
import logging
import shutil


import fastoad.api as oad
import fastga_he.api as oad_he

from fastga_he.gui.power_train_network_viewer import power_train_network_viewer
from fastga_he.gui.analysis_and_plots import (
    aircraft_geometry_plot,
)
from fastga_he.gui.power_train_weight_breakdown import (
    power_train_mass_breakdown,
)
from fastga_he.gui.payload_range import (
    payload_range_outer,
)
DATA_FOLDER_PATH = "data"

WORK_FOLDER_PATH = "workdir"

CONFIGURATION_FILE = pth.join(DATA_FOLDER_PATH, "CaravanDEP_retrofit.yml")
#PT_FILE = pth.join(WORK_FOLDER_PATH, "simple_assembly.yml")
SOURCE_FILE = pth.join(DATA_FOLDER_PATH, "input_CaravanDEP.xml")
RESULTS_FOLDER_PATH = "results"

# For having log messages on screen
logging.basicConfig(level=logging.WARNING, format="%(levelname)-8s: %(message)s")

In [ ]:
# We copy all the useful file in the workdir

shutil.copy(pth.join(DATA_FOLDER_PATH, "mission_vector.yml"), CONFIGURATION_FILE)
shutil.copy(pth.join(DATA_FOLDER_PATH, "simple_assembly.yml"), PT_FILE)

In [ ]:
pt_file_path = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit_2.yml")
network_file_path = pth.join(RESULTS_FOLDER_PATH, "CaravanDEP_powertrain_retrofit_2.html")
power_train_network_viewer(pt_file_path, network_file_path)

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

In [ ]:
oad.variable_viewer(oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True))

In [ ]:
from fastoad import api as api_cs25
from IPython.display import IFrame

N2_FILE = pth.join(DATA_FOLDER_PATH, "n2.html")
api_cs25.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)

IFrame(src=N2_FILE, width="100%", height="500px")

Now we run the problem

In [ ]:
eval_problem = oad.evaluate_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:
pt_file_path = pth.join(DATA_FOLDER_PATH, "CaravanDEP_powertrain_retrofit.yml")
power_train_mass_breakdown(eval_problem.output_file_path, pt_file_path)

In [ ]:
# find_bad_vars.py
import re, os, sys
import numpy as np

FNAME = "solver_errors.0.out"
if not os.path.exists(FNAME):
    # tenta achar o mais recente
    import glob
    cands = sorted(glob.glob("solver_errors*.out"), key=os.path.getmtime)
    if cands:
        FNAME = cands[-1]
    else:
        print("Arquivo solver_errors*.out não encontrado no diretório:", os.getcwd())
        sys.exit(1)

text = open(FNAME, "r", encoding="utf-8", errors="replace").read()
print("Usando arquivo:", FNAME)

# regex encontra 'key': array(
pattern = re.compile(r"(?P<quote>['\"])(?P<key>.+?)(?P=quote)\s*:\s*array\s*\(", flags=re.MULTILINE)

bad = []
found_specific = {"efficiency": [], "shaft_power_out": [], "power": []}
total = 0

for m in pattern.finditer(text):
    total += 1
    key = m.group("key")
    start = m.end()
    # achar fechamento correspondente de parênteses
    depth = 1
    i = start
    N = len(text)
    while i < N and depth > 0:
        ch = text[i]
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        i += 1
    arr_block = text[m.start():i]
    arr_text = arr_block.split(":",1)[1].strip()
    # preparar para eval: transformar array( -> np.array( e nan/inf
    arr_text2 = arr_text.replace("array(", "np.array(")
    arr_text2 = re.sub(r"\bnan\b", "np.nan", arr_text2, flags=re.IGNORECASE)
    arr_text2 = re.sub(r"\binf\b", "np.inf", arr_text2, flags=re.IGNORECASE)
    try:
        val = eval(arr_text2, {"np": np})
    except Exception:
        # fallback: extrair números/np.nan/np.inf
        inner = re.sub(r'^\s*np\.array\s*\(', '', arr_text2)
        if inner.endswith(')'):
            inner = inner[:-1]
        tokens = re.findall(r"np\.nan|np\.inf|[-+]?\d*\.\d+|\d+|[-+]?\d+\.\d+[eE][-+]?\d+|[-+]?\d+[eE][-+]?\d+", inner)
        vals = []
        for t in tokens:
            if t.lower() == "np.nan":
                vals.append(np.nan)
            elif t.lower() == "np.inf":
                vals.append(np.inf)
            else:
                try:
                    vals.append(float(t))
                except:
                    pass
        val = np.array(vals) if vals else None

    if val is None:
        continue
    arr = np.asarray(val)
    if arr.size == 0:
        continue
    mask = ~np.isfinite(arr)
    if np.any(mask):
        idx = np.where(mask)[0][:30].tolist()
        sample = arr[idx].tolist()
        bad.append((key, arr.shape, len(idx), idx[:10], sample[:10]))
    # armazena casos com nomes sugestivos
    lname = key.lower()
    if "efficiency" in lname or "efficiency" in key:
        found_specific["efficiency"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))
    if "shaft_power_out" in lname or "shaft_power_out" in key:
        found_specific["shaft_power_out"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))
    if "power" in lname and "active" in lname or "power" in lname:
        found_specific["power"].append((key, arr.shape, np.isnan(arr).sum(), np.isinf(arr).sum()))

# imprime resumo
print(f"Total de entradas analisadas (padrão 'array('): {total}")
print("Encontradas variáveis com NaN/Inf:", len(bad))
if len(bad) > 0:
    print("\n== Top 60 variáveis com não-finitos (nome | shape | n_nonfinite | índices sample | valores sample):")
    for item in bad[:60]:
        print(item)

print("\n== Variáveis relacionadas (efficiency / shaft_power_out / power) encontradas:")
for k,v in found_specific.items():
    if v:
        print(f"\n{k}:")
        for t in v:
            print("  ", t)
    else:
        print(f"\n{k}: Nenhuma encontrada")

# salvar lista curta
with open("bad_vars_quick.txt", "w", encoding="utf-8") as f:
    for item in bad:
        f.write(f"{item}\n")
print("\nRelatório salvo em bad_vars_quick.txt")


In [ ]:
from openmdao.core.problem import Problem as OMP

_orig_setup = OMP.setup

def _patched_setup(self, *args, **kwargs):
    ret = _orig_setup(self, *args, **kwargs)
    # tenta setar as opções de debug assim que o model existir
    try:
        if hasattr(self, 'model') and self.model is not None:
            # opções comuns
            try:
                self.model.nonlinear_solver.options['iprint'] = 2
                self.model.nonlinear_solver.options['debug_print'] = True
                self.model.nonlinear_solver.options['err_on_non_converge'] = True
            except Exception:
                # se o sistema não tiver nonlinear_solver padrão, percorre subsistemas
                for s in self.model.system_iter(include_self=True, recurse=True):
                    try:
                        s.nonlinear_solver.options['iprint'] = 2
                        s.nonlinear_solver.options['debug_print'] = True
                    except Exception:
                        pass
            # print detalhado de solver
            try:
                self.model.set_solver_print(level=2, depth=3, type_='NL')
            except Exception:
                pass
    except Exception:
        pass
    return ret

# aplica o monkeypatch
OMP.setup = _patched_setup

# agora chama sua função normalmente — quando o Problem for criado + setup(), as opções serão aplicadas
eval_problem = oad.evaluate_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:

# inspect_solver_out.py
import os
import re
import sys
import numpy as np
from pprint import pprint

FNAME = "solver_errors.0.out"

if not os.path.exists(FNAME):
    print("Arquivo não encontrado:", FNAME)
    sys.exit(1)

# lê o arquivo como texto (substitui bytes inválidos)
with open(FNAME, "r", encoding="utf-8", errors="replace") as f:
    text = f.read()

# ------------------------------------------------------------
# 1) tentativa direta: transformar 'array(' -> 'np.array(' e normalizar nan/inf
# ------------------------------------------------------------
transformed = text

# substitui array( -> np.array(
transformed = transformed.replace("array(", "np.array(")

# substitui 'nan' isolado por 'np.nan', sem tocar em 'np.nan' já existentes
transformed = re.sub(r'(?<!np\.)\bnan\b', 'np.nan', transformed, flags=re.IGNORECASE)

# substitui 'inf' isolado por 'np.inf', sem tocar em 'np.inf' já existentes
transformed = re.sub(r'(?<!np\.)\binf\b', 'np.inf', transformed, flags=re.IGNORECASE)

# tenta avaliar o dict inteiro (pode falhar em arquivos com comentários ou sintaxe extra)
obj = None
ok = False
print("Tentando eval do arquivo transformado (pode demorar um pouco)...")
try:
    obj = eval(transformed, {"np": np})
    if isinstance(obj, dict):
        print("Eval completo funcionou: arquivo convertido em dict com", len(obj), "chaves top-level.")
        ok = True
    else:
        print("Eval devolveu tipo:", type(obj))
except Exception as e:
    print("Eval completo falhou:", e)

# ------------------------------------------------------------
# 2) fallback: extrair pares 'nome': array(...) com parsing tolerante
# ------------------------------------------------------------
if not ok:
    print("Fazendo fallback: extraindo pares 'nome': array(...) com regex/parsing tolerante.")
    extracted = {}

    # regex que encontra ocorrências de 'key': array( (captura a posição da chave)
    pattern = re.compile(r"(?P<quote>['\"])(?P<key>.+?)(?P=quote)\s*:\s*array\s*\(", flags=re.MULTILINE)

    for m in pattern.finditer(text):
        key = m.group("key")
        start = m.end()  # posição logo após "array("
        # varrer para encontrar o parêntese de fechamento correspondente
        depth = 1
        i = start
        N = len(text)
        # percorre caractere a caractere até fechar todos os parênteses do array(...)
        while i < N and depth > 0:
            ch = text[i]
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            i += 1
        arr_block = text[m.start():i]  # bloco contendo "'key': array(...)" (ou parte)
        # extrai apenas a parte depois dos dois-pontos
        try:
            arr_text_eval = arr_block.split(":", 1)[1].strip()
        except Exception:
            arr_text_eval = arr_block

        # transforma array( -> np.array( e normaliza nan/inf (usando formas seguras)
        arr_text_eval = arr_text_eval.replace("array(", "np.array(")
        arr_text_eval = re.sub(r'(?<!np\.)\bnan\b', 'np.nan', arr_text_eval, flags=re.IGNORECASE)
        arr_text_eval = re.sub(r'(?<!np\.)\binf\b', 'np.inf', arr_text_eval, flags=re.IGNORECASE)

        # tenta eval seguro do array (namespace limitado)
        val = None
        try:
            val = eval(arr_text_eval, {"np": np})
        except Exception:
            # tentativa de fallback simplificado: extrair números e np.nan/np.inf por regex
            inner = arr_text_eval
            # remover "np.array(" prefix e parênteses finais se existirem
            inner = re.sub(r'^\s*np\.array\s*\(', '', inner)
            if inner.endswith(')'):
                inner = inner[:-1]
            # agora procurar tokens numéricos e np.nan / np.inf
            tokens = re.findall(r"np\.nan|np\.inf|[-+]?\d*\.\d+|\d+|[-+]?\d+\.\d+[eE][-+]?\d+|[-+]?\d+[eE][-+]?\d+", inner)
            vals = []
            for t in tokens:
                if t.lower() == "np.nan":
                    vals.append(np.nan)
                elif t.lower() == "np.inf":
                    vals.append(np.inf)
                else:
                    try:
                        vals.append(float(t))
                    except Exception:
                        # ignora tokens não convertíveis
                        pass
            val = np.array(vals) if vals else None

        extracted[key] = val

    obj = extracted
    print("Fallback extraiu", len(extracted), "entradas.")

# ------------------------------------------------------------
# 3) analisar o dict/extraído e gerar relatório com variáveis que têm NaN/Inf
# ------------------------------------------------------------
def analyze_dict(D):
    bad = []
    if not isinstance(D, dict):
        return bad
    for k, v in D.items():
        try:
            arr = np.asarray(v)
            if arr.size > 0:
                mask = ~np.isfinite(arr)
                if np.any(mask):
                    idx = np.where(mask)[0][:20].tolist()
                    sample_vals = arr[idx].tolist()
                    bad.append((k, arr.shape, idx, sample_vals))
        except Exception:
            # tenta avaliar se é um scalar não-finite
            try:
                fv = float(v)
                if not np.isfinite(fv):
                    bad.append((k, (), [], fv))
            except Exception:
                pass
    return bad

bad_list = analyze_dict(obj)

if bad_list:
    print("\n== Variáveis com NaN/Inf detectadas:", len(bad_list))
    with open("bad_vars.txt", "w", encoding="utf-8") as fout:
        for name, shape, idx_list, sample_vals in bad_list:
            line = f"{name} | shape={shape} | indices_sample={idx_list} | sample_bad_values={sample_vals}\n"
            fout.write(line)
            print(line.strip())
    print("\nRelatório salvo como bad_vars.txt")
else:
    print("Nenhuma variável com NaN/Inf detectada (ou nada foi extraído).")

We can now use the API to create graphs based on the data saved during mission computation

In [ ]:
import os.path as pth

import fastoad.api as oad
import fastga.utils.postprocessing.post_processing_api as api_plots

# For using all screen width
#from IPython.core.display import display, HTML

display(HTML("<style>.container { width:95% !important; }</style>"))

fig = api_plots.mass_breakdown_sun_plot(eval_problem.output_file_path)
fig.show()

In [ ]:
RESULTS_FOLDER_PATH = "results"
MISSION_DATA_FILE = pth.join(RESULTS_FOLDER_PATH, "turbo_electric_propulsion.csv")
PT_DATA_FILE = pth.join(RESULTS_FOLDER_PATH, "turbo_electric_propulsion_pt_watcher.csv")

perfo_viewer = oad_he.PerformancesViewer(
    power_train_data_file_path=PT_DATA_FILE,
    mission_data_file_path=MISSION_DATA_FILE,
    plot_height=800,
)

# Uncomment next lines if you want raw data
# pd.set_option('display.max_rows', 500)
# pd.set_option('display.max_columns', 500)
# pd.set_option('display.width', 200)
# print(perfo_viewer.data)

In [ ]:
oad.variable_viewer(eval_problem.output_file_path)

Let us try optimizing the propeller to get better efficiencies.

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)
oad.optimization_viewer(CONFIGURATION_FILE)

Uncomment next cell if you want to run an optimization, does not seems to work very well with Cobyla (even worse with SLSQP).

In [ ]:
# optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

In [ ]:
oad.optimization_viewer(CONFIGURATION_FILE)